In [0]:
%run ./operations

In [0]:
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StringType
from pyspark.sql import Row
import uuid
from datetime import datetime


In [0]:
# ─── CONFIG ──────────────────────────────────────────────────────────────
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
team_name = "team_lemma"
catalog_name = f"charles_schwab_retailbrokerage_dev_{team_name}"
landing_schema = "landing"
bronze_schema = "bronze"
landing_base_path = f"/Volumes/{catalog_name}/{landing_schema}/PWG/"

In [0]:
# We keep the DOMAIN_MAP from raw_to_landing to know which tables and batches belong to which domain.
# For Bronze ingestion, we only strictly need the 'landing_table' and 'batches' lists.
DOMAIN_MAP = {
    "control": [
        {"landing_table": "batchdate", "batches": ["1", "2", "3"]},
    ],
    "cross": [
        {"landing_table": "date", "batches": ["1"]},
        {"landing_table": "time", "batches": ["1"]},
        {"landing_table": "statustype", "batches": ["1"]},
        {"landing_table": "taxrate", "batches": ["1"]},
        {"landing_table": "industry", "batches": ["1"]},
        {"landing_table": "tradetype", "batches": ["1"]},
    ],
    "market": [
        {"landing_table": "finwire", "batches": ["1"]},
        {"landing_table": "dailymarket", "batches": ["1", "2", "3"]},
    ],
    "hr_broker": [
        {"landing_table": "hr", "batches": ["1"]},
    ],
    "customer": [
        {"landing_table": "customermgmt", "batches": ["1"]},
        {"landing_table": "customer", "batches": ["2", "3"]},
        {"landing_table": "prospect", "batches": ["1", "2", "3"]},
        {"landing_table": "watchhistory", "batches": ["1", "2", "3"]},
    ],
    "account": [
        {"landing_table": "account", "batches": ["2", "3"]},
        {"landing_table": "cashtransaction", "batches": ["1", "2", "3"]},
    ],
    "trade": [
        {"landing_table": "trade", "batches": ["1", "2", "3"]},
        {"landing_table": "trade_history", "batches": ["1"]},
        {"landing_table": "holdinghistory", "batches": ["1", "2", "3"]},
    ],
}

# COMMAND ----------

# ─── MAIN FUNCTION ───────────────────────────────────────────────────────
def load_domain_to_bronze(domain_name: str, spark):
    """
    Reads data from Landing Parquet volume and appends to Bronze Delta tables.
    Bronze Iron Rule: ALL columns are STRING and NEVER delete or update rows.
    """
    domain_name = domain_name.lower().strip()
    if domain_name not in DOMAIN_MAP:
        raise ValueError(
            f"Invalid domain '{domain_name}'. Valid: {list(DOMAIN_MAP.keys())}"
        )

    # Make sure we use the correct catalog and schema for Bronze
    spark.sql(f"USE CATALOG {catalog_name};")
    spark.sql(f"USE SCHEMA {bronze_schema};")
    
    recon_results = []
    
    # Audit columns that should NOT be cast to STRING (or are already handled)
    # Note: _batch_id, _source_name, _source_file, _run_id are carried over as STRING.
    # _ingest_ts is added as TIMESTAMP.
    audit_columns = ["_source_name", "_source_file", "_batch_id", "_run_id", "_ingest_ts"]

    for entry in DOMAIN_MAP[domain_name]:
        table_name = entry["landing_table"]
        
        for batch_id in entry["batches"]:
            batch_folder = f"Batch{batch_id}"
            
            # The parquet path where raw_to_landing wrote the file
            source_path = f"{landing_base_path}{batch_folder}/{table_name}"
            # The Delta table target in Bronze
            target_table = f"{catalog_name}.{bronze_schema}.{table_name}"
            
            try:
                # 1. READ from Landing layer
                df = spark.read.parquet(source_path)
                source_count = df.count()
                
                # 2. DROP _ingestion_ts (which was the landing timestamp)
                if "_ingestion_ts" in df.columns:
                    df = df.drop("_ingestion_ts")
                
                # 3. ADD _ingest_ts (bronze timestamp)
                df = df.withColumn("_ingest_ts", current_timestamp())
                
                # 4. CAST all data columns to STRING (Bronze Iron Rule)
                # Keep audit columns intact
                for col_name in df.columns:
                    if col_name not in audit_columns:
                        df = df.withColumn(col_name, col(col_name).cast(StringType()))
                
                # 5. WRITE to Bronze as Delta using append mode
                df.write \
                  .format("delta") \
                  .mode("append") \
                  .option("mergeSchema", "true") \
                  .partitionBy("_batch_id") \
                  .saveAsTable(target_table)
                
                # Validate the target count after append
                # Note: Because bronze is append-only across batches, target_count will 
                # reflect the entire history of the table up to this point.
                target_count = spark.read.table(target_table).count()
                carried_run_id = df.select("_run_id").first()[0]
                
                recon_results.append(Row(
                    domain=domain_name,
                    table=table_name, 
                    batch_id=batch_folder,
                    source_landing_count=source_count,
                    target_bronze_total=target_count, 
                    status="SUCCESS",
                    run_id=carried_run_id
                ))
                print(f"{table_name} ({batch_folder}) -> APPENDED rows={source_count} | Total in Bronze={target_count}")

            except Exception as e:
                recon_results.append(Row(
                    domain=domain_name,
                    table=table_name, 
                    batch_id=batch_folder,
                    source_landing_count=None,
                    target_bronze_total=None, 
                    status=f"ERROR: {str(e)[:200]}",
                    run_id=None
                ))
                print(f"Error {table_name} ({batch_folder}): {str(e)}")

    recon_df = spark.createDataFrame(recon_results)
    
    # ─── OPERATIONS LAYER LOGGING ───────────────────────────────────────────
    for row in recon_results:
        if row.status == "SUCCESS":
            # Since Bronze is append-only, the rows appended match the source count
            rows_appended = row.source_landing_count
            
            log_pipeline_recon(
                spark=spark,
                run_id=row.run_id,
                batch_id=row.batch_id,
                domain=row.domain.upper(),
                table_name=row.table,
                source_layer="landing",
                target_layer="bronze",
                source_count=row.source_landing_count,
                target_count=rows_appended
            )
            
            log_audit_event(
                spark=spark,
                run_id=row.run_id,
                batch=row.batch_id,
                layer="bronze",
                table_name=row.table,
                operation="APPEND",
                rows_affected=rows_appended
            )
    # ─────────────────────────────────────────────────────────────────────────

    display(recon_df)
    return recon_df
